# SPoRC Dataset Preprocessing — Full Pipeline

**Steps Overview:**
1. Download SPoRC dataset  
2. Decompress `.jsonl.gz` files  
3. Preview dataset structure  
4. Convert speakerTurnData to CSV  
5. Check role distribution  
6. Filter host/guest roles  
7. Check empty / short utterances  
8. Remove empty or meaningless text entries

In [3]:
import os
import pandas as pd
import numpy as np
import gzip
import json
from tqdm import tqdm
from datasets import load_dataset

# Define working directory
PROJECT_DIR = "/Users/allin1307/Desktop/semester 3/NLP/project"
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Working directory: {PROJECT_DIR}")

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Working directory: /Users/allin1307/Desktop/semester 3/NLP/project


In [10]:
# ==========================================================
# Step 1: Download dataset from Hugging Face (auto local check)
# ==========================================================

episodes_path = os.path.join(PROJECT_DIR, "episodeLevelData.jsonl.gz")
turns_path = os.path.join(PROJECT_DIR, "speakerTurnData.jsonl.gz")

# --- Check if local files exist ---
if os.path.exists(episodes_path) and os.path.exists(turns_path):
    print(f"Found existing dataset files, skipping download.\n"
          f"Using local copies:\n{episodes_path}\n{turns_path}")
else:
    print("⬇️  Local files not found. Downloading from Hugging Face...")
    dataset = load_dataset("blitt/SPoRC")
    dataset["episodes"].to_json(episodes_path, orient="records", lines=True)
    dataset["turns"].to_json(turns_path, orient="records", lines=True)
    print(f"Downloaded to:\n{episodes_path}\n{turns_path}")

Found existing dataset files, skipping download.
Using local copies:
/Users/allin1307/Desktop/semester 3/NLP/project/episodeLevelData.jsonl.gz
/Users/allin1307/Desktop/semester 3/NLP/project/speakerTurnData.jsonl.gz


In [9]:
# ==========================================================
# Step 2: Decompress gzipped JSONL files into plain .jsonl
# ==========================================================

#episodes_jsonl = episodes_path.replace(".gz", "")
turns_jsonl = turns_path.replace(".gz", "")

def decompress_gz(gz_path):
    jsonl_path = gz_path.replace(".gz", "")
    with gzip.open(gz_path, "rb") as f_in, open(jsonl_path, "wb") as f_out:
        f_out.write(f_in.read())
    print(f"Decompressed: {os.path.basename(jsonl_path)}")
    return jsonl_path

if os.path.exists(turns_jsonl):
    print(f"Found decompressed JSONL files, skipping decompression.")
else:
    episodes_jsonl = decompress_gz(episodes_path)
    turns_jsonl = decompress_gz(turns_path)

Found decompressed JSONL files, skipping decompression.


In [21]:
# ==========================================================
# Step 3: Preview structure of speaker-turn-level data
# ==========================================================
turns_gz_path = os.path.join(PROJECT_DIR, turns_path)
print("Previewing file:", turns_gz_path)

# Read first 2 JSON lines from the compressed file
with gzip.open(turns_gz_path, "rt", encoding="utf-8") as f:
    first_lines = []
    for i in range(2):
        try:
            first_lines.append(json.loads(next(f)))
        except StopIteration:
            break
        except json.JSONDecodeError as e:
            print("JSON decode error on line", i, ":", e)
            break

print("Columns:", list(first_lines[0].keys()))
display(pd.DataFrame(first_lines))

Previewing file: /Users/allin1307/Desktop/semester 3/NLP/project/speakerTurnData.jsonl.gz
Columns: ['mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27.5Hz_sma3nzMean', 'F1frequency_sma3nzMean', 'turnText', 'speaker', 'startTime', 'endTime', 'duration', 'mp3url', 'turnCount', 'inferredSpeakerRole', 'inferredSpeakerName']


,mfcc1_sma3Mean,mfcc2_sma3Mean,mfcc3_sma3Mean,mfcc4_sma3Mean,F0semitoneFrom27.5Hz_sma3nzMean,F1frequency_sma3nzMean,turnText,speaker,startTime,endTime,duration,mp3url,turnCount,inferredSpeakerRole,inferredSpeakerName
0,20.544348,13.441536,16.438079,3.834616,13.972970,717.300994,I'm Simon Shapiro and this is Sing Out Speak ...,[SPEAKER_00],0.0,60.00,60.00,https://www.buzzsprout.com/783020/4252475-best...,1,host,Simon Shapiro
1,17.601921,5.544924,17.161562,3.468099,16.826481,735.535396,I'm Simon Shapiro and this is Sing Out Speak ...,[SPEAKER_00],0.0,78.16,78.16,https://www.buzzsprout.com/783020/4165286-it-s...,1,host,Simon Shapiro


In [22]:
# ==========================================================
# Step 4: Convert speakerTurnData.jsonl.gz to CSV
# ==========================================================

csv_path = os.path.join(PROJECT_DIR, "sporc_turns_selected.csv")

if os.path.exists(csv_path):
    print("Found existing CSV file, skipping conversion.")
    print("Using:", csv_path)
else:
    # Read directly from compressed JSONL.GZ to save memory
    print("Converting speakerTurnData.jsonl.gz to CSV...")
    with gzip.open(os.path.join(PROJECT_DIR, "speakerTurnData.jsonl.gz"), "rt", encoding="utf-8") as f_in:
        lines = [json.loads(line) for line in tqdm(f_in, desc="Converting")]
        df_turns = pd.DataFrame(lines)
        df_turns.to_csv(csv_path, index=False)
    print("Saved as CSV:", csv_path)
    del df_turns
    gc.collect()

Found existing CSV file, skipping conversion.
Using: /Users/allin1307/Desktop/semester 3/NLP/project/sporc_turns_selected.csv


In [24]:
# ==========================================================
# Step 5: Check and display role counts
# ==========================================================

turns_gz_path = os.path.join(PROJECT_DIR, turns_path)
print("Reading from:", turns_gz_path)

# Load inferredSpeakerRole column only
roles = []
with gzip.open(turns_gz_path, "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        roles.append(record.get("inferredSpeakerRole", None))
        if i > 200000:  # limit to first 200k lines for speed
            break

df_roles = pd.DataFrame(roles, columns=["inferredSpeakerRole"])
role_counts = df_roles["inferredSpeakerRole"].value_counts(dropna=False)

print("Role distribution BEFORE filtering:\n", role_counts)
print("\nUnique role labels:", df_roles["inferredSpeakerRole"].unique())

Reading from: /Users/allin1307/Desktop/semester 3/NLP/project/speakerTurnData.jsonl.gz
Role distribution BEFORE filtering:
 inferredSpeakerRole
NO_INFERRED_ROLE    179243
host                 15802
guest                 4957
Name: count, dtype: int64

Unique role labels: ['host' 'NO_INFERRED_ROLE' 'guest']


In [31]:
# ==========================================================
# Step 6: Filter host/guest roles only
# ==========================================================
out_filtered = os.path.join(PROJECT_DIR, "sporc_turns_selected_clean.csv")

if os.path.exists(out_filtered):
    print("Found existing filtered file:", out_filtered)
    filtered = pd.read_csv(out_filtered)
    role_col = "inferredSpeakerRole" if "inferredSpeakerRole" in df_filtered.columns else "role"
    print("\nRole distribution AFTER filtering:\n", df_filtered[role_col].value_counts())
else:
    df_full = pd.read_csv(csv_path)

    # Keep only host / guest
    keep_labels = {"host", "guest"}
    filtered = df_full[df_full["inferredSpeakerRole"].isin(keep_labels)].copy()
    removed = len(df_full) - len(filtered)

    filtered.to_csv(out_filtered, index=False)
    print(f"Filtered roles: kept {len(filtered):,}, removed {removed:,}")
    print(f"Saved → {out_filtered}")


Found existing filtered file: /Users/allin1307/Desktop/semester 3/NLP/project/sporc_turns_selected_clean.csv

Role distribution AFTER filtering:
 role
host     5810050
guest    1452828
Name: count, dtype: int64


In [36]:
# ==========================================================
# Step 7: Count empty and short (<10 chars) utterances
# ==========================================================
empty_count = filtered["text"].isna().sum()
short_count = (filtered["text"].fillna("").str.len() < 10).sum()
print(f"Total: {len(filtered):,}")
print(f"Empty utterances: {empty_count:,}")
print(f"Short utterances (<10 chars): {short_count:,}")

Total: 7,262,878
Empty utterances: 8,058
Short utterances (<10 chars): 751,454


In [41]:
# ==========================================================
# Step 8: Remove empty and meaningless short texts
# ==========================================================
df = filtered.copy()

# Remove NaN / empty / whitespace-only
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""]
df = df[~df["text"].str.fullmatch(r"[\.,!\?\-–—\s]+")]

# Remove meaningless short texts
meaningless = {
    "ok", "okay", "yeah", "yep", "no", "nah", "hmm", "uh", "um",
    "right", "sure", "mm", "mmm", "ha", "haha", "yes", "wow", "oh"
}
df = df[~df["text"].str.lower().isin(meaningless)]
df = df[df["text"].str.len() >= 10]

output_final = os.path.join(PROJECT_DIR, "sporc_final_clean_min.csv")
df.to_csv(output_final, index=False)
print("Final cleaned dataset saved:", output_final)

Final cleaned dataset saved: /Users/allin1307/Desktop/semester 3/NLP/project/sporc_final_clean_min.csv
